In [27]:
%%capture
!pip install facenet-pytorch

In [28]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torchsummary import summary
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter

import numpy as np
import pandas as pd
import os
%matplotlib inline
import matplotlib.pyplot as plt
import glob
import re

import PIL
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True

workers = 0 if os.name == 'nt' else 4

## Load models

In [29]:
# set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Current device: {}'.format(device))

Current device: cuda:0


In [30]:
# Load model
'''
This model is used to detect faces and it returns the face (cropped)

image_size: output image size
margin: margin of the bounding box added in the ouput image
min_face_size: minimum face size to search within the image
'''
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device
)

In [31]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [32]:
# # model architecture and summary
# summary(resnet, (3, 160, 160))

## Load Data

In [33]:
import os
os.getcwd()

'/home/pj00/projects/Github/online_face_recognition_and_detection/DevelopmentNotebooks'

In [34]:
# !rmdir /content/data/test_images/.ipynb_checkpoints

In [35]:
def collate_fn(x):
    return x[0]

In [36]:
dataset = datasets.ImageFolder('/home/pj00/projects/Github/online_face_recognition_and_detection/Data/test_images')
dataset.idx_to_class = {i:c for c, i in dataset.class_to_idx.items()}
loader = DataLoader(dataset, collate_fn=collate_fn, num_workers=workers)

## Face detection using MTCNN

In [37]:
aligned = []
names = []
for x, y in loader:
    x_aligned, prob = mtcnn(x, return_prob=True)
    if x_aligned is not None:
        print('Face detected with probability: {:8f}'.format(prob))
        aligned.append(x_aligned)
        names.append(dataset.idx_to_class[y])


Face detected with probability: 0.999983
Face detected with probability: 0.999934
Face detected with probability: 0.999733
Face detected with probability: 0.999876
Face detected with probability: 0.999992


## Calculate image embedding

In [38]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned).detach().cpu()

In [39]:
# embeddings distance
dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
print(pd.DataFrame(dists, columns=names, index=names))

                angelina_jolie  bradley_cooper  kate_siegel  paul_rudd  \
angelina_jolie        0.000000        0.159367     0.249892   0.323930   
bradley_cooper        0.159367        0.000000     0.234019   0.348651   
kate_siegel           0.249892        0.234019     0.000000   0.411586   
paul_rudd             0.323930        0.348651     0.411586   0.000000   
shea_whigham          0.158514        0.186108     0.321208   0.346829   

                shea_whigham  
angelina_jolie      0.158514  
bradley_cooper      0.186108  
kate_siegel         0.321208  
paul_rudd           0.346829  
shea_whigham        0.000000  


## Finetuning on a new image

In [40]:
data_dir = '/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images'
workers = 0 if os.name == 'nt' else 8
random_people = len(glob.glob(os.path.join('/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images/random_people/*.jpg')))

print('There are {} random people.'.format(random_people))

There are 0 random people.


#### Extract the face from the training dataset

In [41]:
# !rmdir /content/data/train_images/.ipynb_checkpoints

In [42]:
dataset = datasets.ImageFolder(data_dir, transform=transforms.Resize((512, 512)))
dataset.samples = [
    (p, p.replace(data_dir, data_dir + '_cropped'))
        for p, _ in dataset.samples
]

loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=1,
    collate_fn=training.collate_pil
)

In [43]:
for i, (x, y) in enumerate(loader):
    mtcnn(x, save_path=y)
    print('\rBatch {} of {}'.format(i + 1, len(loader)), end='')

Batch 5 of 5

#### Data Augmentation

In [44]:
aug_transform = transforms.Compose([
    v2.ToTensor(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=.5, hue=.3),
    v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)), v2.RandomAutocontrast(),
    v2.ToPILImage()])

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/torchvision/transforms/v2/_deprecated.py:41: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.
  warnings.warn(


In [45]:
new_path = y[0].replace('train_images_cropped', 'train_images_augmented')
new_path

'/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images_augmented/1/1.png'

In [46]:
new_path = re.sub(r'(\d+)\.jpg', r'{}.jpg'.format(5), new_path)
new_path

'/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images_augmented/1/1.png'

In [47]:
try:
    os.mkdir('/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images_augmented')
except:
    print('Path exists')

Path exists


In [48]:
num_augs = 64
random_people_counter = 0
for _, y in loader:
    im = Image.open(y[0])
    try:
        os.mkdir('/'.join(y[0].replace('train_images_cropped', 'train_images_augmented').split('/')[:-1]))
    except:
        print('Directory already exists')
    if 'random_people' in y[0]:
        for _ in range(num_augs//random_people):
            new_im = aug_transform(im)
            new_path = y[0].replace('train_images_cropped', 'train_images_augmented')
            new_path = re.sub(r'(\d+)\.jpg', r'{}.jpg'.format(random_people_counter), new_path)
            # new_im.save(y[0].replace('train_images_cropped', 'train_images_augmented').replace('*.jpg', '{}.jpg'.format(random_people_counter)))
            new_im.save(new_path)
            random_people_counter += 1
    else:
        for i in range(num_augs):
            new_im = aug_transform(im)
            new_im.save(y[0].replace('train_images_cropped', 'train_images_augmented').replace('1.jpg', '{}.jpg'.format(i)))


Directory already exists
Directory already exists
Directory already exists
Directory already exists
Directory already exists


#### Finetune the model

In [49]:
data_dir = '/home/pj00/projects/Github/online_face_recognition_and_detection/Data/train_images_augmented'

batch_size = 16
epochs = 8

resnet = InceptionResnetV1(
    classify=True,
    pretrained='vggface2',
    num_classes=len(dataset.class_to_idx) if len(dataset.class_to_idx)>1 else 2
).to(device)

print('Model is trained on {} classes.'.format(len(dataset.class_to_idx)))

Model is trained on 2 classes.


In [50]:
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

# trans = transforms.Compose([
#     np.float32,
#     transforms.ToTensor(),
#     fixed_image_standardization
# ])

trans = transforms.Compose([
    np.float32,
    transforms.ToTensor()])

dataset = datasets.ImageFolder(data_dir, transform=trans)
img_inds = np.arange(len(dataset))
np.random.shuffle(img_inds)
train_inds = img_inds[:int(0.8 * len(img_inds))]
val_inds = img_inds[int(0.8 * len(img_inds)):]

train_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(train_inds)
)
val_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(val_inds)
)

In [51]:
loss_fn = torch.nn.CrossEntropyLoss()
metrics = {
    'fps': training.BatchTimer(),
    'acc': training.accuracy
}

In [52]:
writer = SummaryWriter()
writer.iteration, writer.interval = 0, 10

print('\n\nInitial')
print('-' * 10)
resnet.eval()
training.pass_epoch(
    resnet, loss_fn, val_loader,
    batch_metrics=metrics, show_running=True, device=device,
    writer=writer
)

for epoch in range(epochs):
    print('\nEpoch {}/{}'.format(epoch + 1, epochs))
    print('-' * 10)

    resnet.train()
    training.pass_epoch(
        resnet, loss_fn, train_loader, optimizer, scheduler,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

    resnet.eval()
    training.pass_epoch(
        resnet, loss_fn, val_loader,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

writer.close()



Initial
----------
Valid |     1/1    | loss:    0.0185 | fps:   44.8100 | acc:    1.0000   

Epoch 1/8
----------
Train |     4/4    | loss:    1.5453 | fps:   83.2472 | acc:    0.4375   
Valid |     1/1    | loss:    1.2000 | fps:   41.5615 | acc:    0.6154   

Epoch 2/8
----------
Train |     4/4    | loss:    0.9147 | fps:   53.8450 | acc:    0.4844   
Valid |     1/1    | loss:    0.1072 | fps:   40.1605 | acc:    1.0000   

Epoch 3/8
----------
Train |     4/4    | loss:    0.9030 | fps:   53.5782 | acc:    0.5469   
Valid |     1/1    | loss:   12.7379 | fps:   38.9334 | acc:    0.0000   

Epoch 4/8
----------
Train |     4/4    | loss:    0.6862 | fps:   53.1192 | acc:    0.5938   
Valid |     1/1    | loss:    0.1510 | fps:   40.5360 | acc:    1.0000   

Epoch 5/8
----------
Train |     4/4    | loss:    0.7150 | fps:   54.2458 | acc:    0.5156   
Valid |     1/1    | loss:    0.2612 | fps:   41.2836 | acc:    0.7692   

Epoch 6/8
----------
Train |     4/4    | loss:    0.7

## Test the model

In [53]:
data_dir = '/home/pj00/projects/Github/online_face_recognition_and_detection/Data/val_images'
dataset = datasets.ImageFolder(data_dir)
dataset.idx_to_class = {i:c for c, i in dataset.class_to_idx.items()}
loader = DataLoader(dataset, collate_fn=collate_fn, num_workers=workers)

In [54]:
image, label = dataset[0]

In [55]:
aligned = []
names = []
for x, y in loader:
    x_aligned, prob = mtcnn(x, return_prob=True)
    if x_aligned is not None:
        print('Face detected with probability: {:8f}'.format(prob))
        aligned.append(x_aligned)
        names.append(dataset.idx_to_class[y])


Face detected with probability: 0.999986
Face detected with probability: 0.996072


In [56]:
resnet.eval()
print('Done')

Done


In [57]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned).detach().cpu()

# embeddings distance
dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
print(pd.DataFrame(dists, columns=names, index=names))

          0         1
0  0.000000  0.943546
1  0.943546  0.000000


## Save weights

In [58]:
current_stages = os.listdir('/home/pj00/projects/Github/online_face_recognition_and_detection/model_weights')
current_max = max(current_stages, key = lambda x: int(x.split('.')[0]))
current_max = '{}.pt'.format(int(current_max.split('.')[0])+1)

In [59]:
torch.save(resnet.state_dict(), '/home/pj00/projects/Github/online_face_recognition_and_detection/model_weights/{}'.format(current_max))

In [60]:
current_max

'9.pt'